[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/surfaces/skinning/skinning.ipynb)

# Linear Blend and Motor Skinning

This example deforms a cylindrical skin over two bones as one bone twists about their shared axis. Each vertex of the skin follows both bones, weighted by how near it lies to each. Mixing the destination points that each bone would carry a vertex to makes a twisted limb collapse in the middle like a wrung-out towel, whereas mixing the rigid motions of the bones themselves keeps every cross-section circular at its rest radius.

In 3D projective geometric algebra, a bone's rigid motion is a motor, and its action on points is the linear map `motor >> Point`. Linear blend skinning blends the two bones' point maps and applies the blended map; motor skinning blends the motors, normalises, and applies the sandwich, or follows the screw geodesic between them through the motor logarithm and exponential. Because the maps and the motors are extensors of the same kind, both blends take the same form, while only the motor blends remain rigid motions.

In the matrix notation of computer graphics the point map reads as a $4\times4$ transformation matrix and linear blend skinning as the weighted matrix sum $((1 - w) M_0 + w M_1)\mathbf p$. In dual quaternion notation a motor reads as a unit dual quaternion and normalised motor blending as dual quaternion linear blending.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import numpy as np
from numga import NumpyContext
from numga.algebras import PGA3D as ga

from examples.util.animation import save_figure
from examples.surfaces.skinning import render
from examples.surfaces.skinning.core import cylinder

np.set_printoptions(precision=4, suppress=True)

mv = NumpyContext(ga).multivector
Point = ga.gatype.antivector()                      # Antivector-
Motor = ga.gatype.rotor()                           # Rotor+

## 1. A cylinder on two bones

The rest skin is a unit cylinder around the x axis from 0 to 1, with each ring weighted toward the tip bone in proportion to its distance along the axis. The root bone stays fixed while the tip bone twists about the shared axis.

In [ ]:
# The cylinder's rings along the bones and vertices around each, and how far the tip bone is twisted.
n_rings, n_around = 12, 24
twist_angle = np.radians(150.0)

In [ ]:
vertices, weights = cylinder(n_rings, n_around)                                # [vertices] Point-, [vertices]
root = mv.rotor()                                                              # [] Motor+
tip = (mv.yz * (twist_angle / 2)).exp()                                        # [] Motor+

## 2. Blending motors and blending point maps

Blending the motors and normalising keeps a rigid motion at every weight; the slerp, the exponential of the weighted logarithm of the relative motor, follows the geodesic between them and spaces the twist evenly; blending the point maps averages two points on a circle and pulls them in towards the chord. From left to right, the figure shows the cylinder deformed by normalised motor blending, motor slerp, and point-map blending.

In [ ]:
# Blend the bone motors linearly and normalise back to the motor manifold.
blended_motors = (root + weights * (tip - root)).normalized()                  # [vertices] Motor+
motor_skin = blended_motors >> vertices                                        # [vertices] Point-

# Slerp follows the screw geodesic: the logarithm of the relative motor, scaled by the
# weight and exponentiated. The normalised linear blend is a chord of the same arc, so both
# are rigid about the axis and differ only in how their twist angles are spaced along x.
slerp_motors = ((tip / root).log() * weights).exp() * root                     # [vertices] Motor+
slerp_skin = slerp_motors >> vertices                                          # [vertices] Point-

# Linear blend skinning blends the two bones' point maps and applies the blended map.
blended_maps = (root >> Point) + weights * ((tip >> Point) - (root >> Point))  # [vertices] Point- <- Point-
matrix_skin = blended_maps(vertices)                                           # [vertices] Point-

save_figure(render.draw_skinning(motor_skin, slerp_skin, matrix_skin, n_rings, n_around), "skinning");

In [ ]:
# Joining a unit vertex with the bone axis mv.yz forms the plane through both, whose norm is the radial distance.
for label, deformed in (("normalised motor blend", motor_skin), ("motor slerp", slerp_skin), ("point-map blend", matrix_skin)):
    radii = (deformed.normalized() & mv.yz).norm()                             # [vertices] Scalar+
    print(f"{label}, minimum radius: {radii.to_array().min():.4f}")

In [ ]:
# checks
from examples.surfaces.skinning import core

skins = (motor_skin, slerp_skin, matrix_skin)
for built, reference in zip(skins, core.blend_skin(vertices, weights, root, tip)):
    np.testing.assert_allclose((built - reference).kernel, 0.0, atol=1e-12)

# Both motor blends keep every vertex at unit radius; the point-map blend pulls a vertex of weight w in to the
# chord between its two images.
np.testing.assert_allclose((motor_skin.normalized() & mv.yz).norm().to_array(), 1.0, atol=1e-12)
np.testing.assert_allclose((slerp_skin.normalized() & mv.yz).norm().to_array(), 1.0, atol=1e-12)
chord_radius = np.sqrt(1 - 2 * weights * (1 - weights) * (1 - np.cos(twist_angle)))
np.testing.assert_allclose((matrix_skin.normalized() & mv.yz).norm().to_array(), chord_radius, atol=1e-12)